# STEP 04: Model Training & Preprocessing Pipeline

This notebook sets up the preprocessing transformers and trains candidate regressor models:
1. Split engineered dataset into 80% Train / 20% Test with random_state=42.
2. Define `ColumnTransformer` with `TargetEncoder` for high-cardinality localities (`Area Locality`, `City`, `City_Locality`, `City_BHK`, `City_Furnishing`) and `StandardScaler` for numerical attributes.
3. Train candidate models: Linear Regression, Random Forest, XGBoost, and Voting Ensemble.

In [2]:
# Import training dependencies
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

# Load engineered dataset
df = pd.read_csv("../dataset/processed/engineered_house_rent_dataset.csv")
print("Engineered Dataset Shape:", df.shape)

Engineered Dataset Shape: (4177, 25)


In [3]:
# Train/Test Split (80/20 split, random_state=42 to prevent data leakage)
X = df.drop(columns=["Rent"])
y = df["Rent"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

Training set shape: (3341, 24), Testing set shape: (836, 24)


In [4]:
# Define ColumnTransformer for numerical and high-cardinality categorical features
# TargetEncoder converts high-cardinality locality names into target-informed numerical representations,
# avoiding sparse 2200+ column one-hot vectors that cripple decision trees.
numeric_features = [
    "BHK", "Size", "Bathroom", "Current_Floor", "Total_Floors",
    "Floor_Ratio", "Bathroom_BHK_Ratio", "Size_Per_BHK", "Size_Per_Bathroom",
    "Is_Top_Floor", "Is_Ground_Floor", "Posted_Year", "Posted_Month", "Posted_DayOfWeek", "Log_Size"
]

categorical_features = [
    "Area Type", "Area Locality", "City", "Furnishing Status", "Tenant Preferred", "Size_Category",
    "City_Locality", "City_BHK", "City_Furnishing"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = TargetEncoder(smooth="auto", cv=5, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
print("Preprocessor defined successfully.")

Preprocessor defined successfully.


In [5]:
# Train selected candidate models with log-target transformation
rf = RandomForestRegressor(n_estimators=800, max_depth=18, min_samples_split=2, min_samples_leaf=1, max_features=0.7, random_state=42, n_jobs=-1)
xgb_m = xgb.XGBRegressor(
    n_estimators=1500, learning_rate=0.012, max_depth=6, subsample=0.75, colsample_bytree=0.7,
    min_child_weight=3, gamma=0.05, reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
)
voting = VotingRegressor(estimators=[('xgb', xgb_m), ('rf', rf)], weights=[2, 1])

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": rf,
    "XGBoost": xgb_m,
    "Voting Ensemble": voting
}

for name, model in candidate_models.items():
    pipe = TransformedTargetRegressor(
        regressor=Pipeline(steps=[("preprocessor", preprocessor), ("model", model)]),
        func=np.log1p,
        inverse_func=np.expm1
    )
    pipe.fit(X_train, y_train)
    print(f"Candidate model '{name}' successfully trained.")

Candidate model 'Linear Regression' successfully trained.
Candidate model 'Random Forest' successfully trained.
Candidate model 'XGBoost' successfully trained.
Candidate model 'Voting Ensemble' successfully trained.
